In [ ]:
import os
import csv
import shutil
import requests
from tqdm import tqdm
from PIL import Image
from io import BytesIO

import aiohttp
import asyncio

In [ ]:
#flowers = {
#    "rose":             "Rosa banksiae",
#    "rose2":            "Rosa pisocarpa",
#    "rose4":            "Rosa chinensis",
#    "rose5":            "Rosa rugosa",
#   "rose6":            "Rosa multiflora",
#    "rose7":            "Rosa gallica",
#    "rose9":            "Rosa damascena",
#    "rose10":           "Rosa canina",
#   "tulip":            "Tulipa gesneriana",



#    "tulip":            "Tulipa",
#    "sunflower":        "Helianthus annuus",
#    "daisy":            "Bellis perennis",
#    "lily":             "Lilium",
#    "peony":            "Paeonia",
#    "dahlia":           "Dahlia",
#    "chrysanthemum":    "Chrysanthemum",
#    "iris":             "Iris",
#    "poppy":            "Papaver rhoeas",
#    "hydrangea":        "Hydrangea",
#    "carnation":        "Dianthus caryophyllus",
#    "freesia":          "Freesia",
#    "gerbera":          "Gerbera jamesonii",
#    "anemone":          "Anemone",
#    "ranunculus":       "Ranunculus",
#    "protea":           "Protea",
#    "bird_of_paradise": "Strelitzia reginae",
#    "lisianthus":       "Eustoma grandiflorum",
#    "gypsophila":       "Gypsophila paniculata",
#    "hellebore":        "Helleborus",
#    "anthurium":        "Anthurium",
#    "allium":           "Allium",
#    "cosmos":           "Cosmos bipinnatus",
#    "zinnia":           "Zinnia elegans",
#    "foxglove":         "Digitalis purpurea",
#    "lavender":         "Lavandula",
#    "wisteria":         "Wisteria",
#}

fruits = {
    "apple":        "Malus domestica",
    "blackberry":    "Rubus fruticosus",
    "orange":       "Citrus sinensis",
    "grape":        "Vitis vinifera",
    "strawberry":   "Fragaria × ananassa",
    "raspberry":    "Rubus idaeus",
    "watermelon":   "Citrullus lanatus",
    "pineapple":    "Ananas comosus",
    "mango":        "Mangifera indica",
    "blueberry":    "Vaccinium angustifolium",
    "peach":        "Prunus persica",
    "kiwi":         "Actinidia deliciosa",
    "pear":         "Pyrus calleryana",
    "plum":         "Prunus domestica",
    "cherry":       "Prunus avium",
    "grapefruit":   "Citrus × paradisi",
    "melon":        "Cucumis melo",
    "papaya":       "Carica papaya",
    "fig":          "Ficus carica",
    "pomegranate":   "Punica granatum",
    "avocado":      "Persea americana",
    "coconut":      "Cocos nucifera",
    "banana":       "Musa",
    "lime":         "Citrus aurantiifolia",
    "lemon":        "Citrus limon",
    "raspberry":    "Rubus idaeus",
    "blackberry":    "Rubus fruticosus",
    "cranberry":    "Vaccinium macrocarpon",
    "passionfruit":  "Passiflora edulis",
    "guava":        "Psidium guajava",
    "lychee":       "Litchi chinensis",
}

In [ ]:
# ---------------------------
# TELECHARGEMENT + RESIZE
# ---------------------------

def download_and_resize(url, filepath):

    try:
        r = requests.get(url, timeout=15)

        img = Image.open(BytesIO(r.content)).convert("RGB")

        img = img.resize(IMAGE_SIZE)

        img.save(filepath, "JPEG", quality=90)

        return True

    except:
        return False

async def download_and_resize_async(session, url, filepath, timeout=10):
    """Download and resize a single image asynchronously."""
    try:
        async with session.get(url, timeout=aiohttp.ClientTimeout(total=timeout)) as resp:
            if resp.status == 200:
                img = Image.open(BytesIO(await resp.read())).convert("RGB")
                img = img.resize(IMAGE_SIZE)
                img.save(filepath, "JPEG", quality=90)
                return True
    except Exception as e:
        print(f"    [ERROR] {str(e)[:60]}", end="")
    return False

async def download_batch_async(urls_filepaths, max_concurrent=10):
    """
    Download multiple images concurrently.
    
    Args:
        urls_filepaths: list of (url, filepath) tuples
        max_concurrent: max parallel downloads (10–20 is good; don't exceed ~50)
    
    Returns:
        count of successful downloads
    """
    success = 0
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def bounded_download(session, url, filepath):
        nonlocal success
        async with semaphore:
            if await download_and_resize_async(session, url, filepath):
                success += 1
            return success
    
    connector = aiohttp.TCPConnector(limit_per_host=5)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [bounded_download(session, url, fp) for url, fp in urls_filepaths]
        await asyncio.gather(*tasks)
    
    return success

In [ ]:
# ---------------------------
# PARAMETRES
# ---------------------------

DATASET_DIR       = "../data/dataset_fruits"
IMAGES_PER_FLOWER = 500   # images to download per flower
IMAGE_SIZE        = (400, 400)
BATCH_SIZE        = 20    # images to download per round

# ---------------------------
# CREATION DOSSIERS
# ---------------------------

os.makedirs(DATASET_DIR, exist_ok=True)
csv_rows = []

for fruit_name, taxon in fruits.items():
    print(f"\n{'='*60}")
    print(f"Downloading: {fruit_name} ({taxon})")

    fruit_dir = os.path.join(DATASET_DIR, fruit_name)
    os.makedirs(fruit_dir, exist_ok=True)

    saved_count   = 0   # images saved so far
    url_idx       = 0   # pointer into all_urls
    page          = 1
    all_urls      = []
    all_dates     = []
    no_more_pages = False

    pbar = tqdm(total=IMAGES_PER_FLOWER, desc=f"  Saving {fruit_name}")

    while saved_count < IMAGES_PER_FLOWER:

        # ── 1. Fetch more URLs from the API until we have a full batch ──────
        while len(all_urls) - url_idx < BATCH_SIZE and not no_more_pages:
            api_url = "https://api.inaturalist.org/v1/observations"
            params  = {
                "taxon_name":    taxon,
                "quality_grade": "casual,needs_id,research",
                "identified":    "true",
                "photos":        "true",
                "per_page":      200,
                "page":          page,
                "year":          "2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",
                "order":         "asc",
            }
            results = requests.get(api_url, params=params).json().get("results", [])
            if not results:
                no_more_pages = True
                break
            for obs in results:
                for photo in obs["photos"]:
                    all_urls.append(photo["url"].replace("square", "large"))
                    all_dates.append(obs.get("observed_on", ""))
            page += 1

        # ── 2. Build the next batch ─────────────────────────────────────────
        batch_end   = min(url_idx + BATCH_SIZE, len(all_urls))
        batch_urls  = all_urls[url_idx:batch_end]
        batch_dates = all_dates[url_idx:batch_end]

        if not batch_urls:
            print(f"  No more URLs available. Saved {saved_count}/{IMAGES_PER_FLOWER}")
            break

        dest_paths = [
            os.path.join(fruit_dir, f"{fruit_name}_{saved_count + i:04d}.jpg")
            for i in range(len(batch_urls))
        ]
        urls_filepaths = list(zip(batch_urls, dest_paths))

        # ── 3. Async download of the batch ──────────────────────────────────
        await download_batch_async(urls_filepaths, max_concurrent=10)

        # ── 4. Record saved images ──────────────────────────────────────────
        for dest_path, date_val in zip(dest_paths, batch_dates):
            if not os.path.exists(dest_path):
                continue
            csv_rows.append([
                fruit_name,
                os.path.join(fruit_name, os.path.basename(dest_path)),
                date_val,
            ])
            saved_count += 1
            pbar.update(1)
            if saved_count >= IMAGES_PER_FLOWER:
                break

        url_idx = batch_end

        if no_more_pages and url_idx >= len(all_urls):
            print(f"  Exhausted all available URLs. Saved {saved_count}/{IMAGES_PER_FLOWER}")
            break

    pbar.close()
    print(f"  ✓ {saved_count}/{IMAGES_PER_FLOWER} images saved for {fruit_name}")

    CSV_FILE = os.path.join(DATASET_DIR, "dataset_fruits.csv")
    with open(CSV_FILE, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["fruit_type", "image_path", "date_observed"])
        writer.writerows(csv_rows)

    print("\nDataset et CSV créés avec succès.")